'''
Source: Impact4cast (Max Planck Institute)
Original code from the Max Planck Institute for Informatics / Max Planck Institute for Security and Privacy research group.

Modifications: Bug fixes for checkpoint resume mechanism and dataset adaptation for scientific entity impact prediction.
'''


In [ ]:
# run_best_predict_no_model_py.py (修复版：改进模型输入维度推断)
import os
import re
import time
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn 
import gc

# ---------------- Config ----------------
BASE_DIR = r"E:\study\research\impact4cast_cscl"
RESULT_FILE = os.path.join(BASE_DIR, "2019_train", "t0_c2_result", "All_AUC_Report_Year_Train_2019.txt")
MODEL_DIR = os.path.join(BASE_DIR, "2019_train", "t0_c2_net")
OUTPUT_DIR = os.path.join(BASE_DIR, "predictions")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 真正的预测特征文件 (141 列归一化特征)
FEATURE_FILE = os.path.join(BASE_DIR, "data_eval", "eval_data_pair_feature.parquet")

# 包含原始 ID 和可能标签的解决方案文件 (包含 v1, v2)
ID_FILE = os.path.join(BASE_DIR, "data_eval", "eval_data_pair_solution.parquet") 

BATCH_SIZE = 2048


# ---------------- 模型定义 ----------------
class ff_network(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ff_network, self).__init__()

        self.semnet = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size)
        )

    def forward(self, x):
        res = self.semnet(x)
        return res

# ---------------- helper functions ----------------
def find_best_ir(result_file_path):
    pattern = re.compile(r'IR=\[(\d+)\]: train=([\d.]+); test=([\d.]+); eval=([\d.]+)')
    results = []
    with open(result_file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            m = pattern.search(line)
            if m:
                ir, train, test, eval_ = m.groups()
                results.append({
                    'IR': int(ir),
                    'train': float(train),
                    'test': float(test),
                    'eval': float(eval_)
                })
    if not results:
        raise RuntimeError(f"未在 {result_file_path} 中解析到任何 IR 记录，请检查文件格式或路径。")
    df = pd.DataFrame(results)
    best_ir = int(df.loc[df['eval'].idxmax(), 'IR'])
    best_eval = float(df['eval'].max())
    return best_ir, best_eval, df


def find_model_file_for_ir(model_dir, ir):
    ir_token = f"IR_{ir:03d}"
    candidates = [f for f in os.listdir(model_dir) if ir_token in f and f.endswith('.pt')]
    if not candidates:
        ir_token2 = f"IR_{ir}"
        candidates = [f for f in os.listdir(model_dir) if ir_token2 in f and f.endswith('.pt')]
    if not candidates:
        raise FileNotFoundError(f"在 {model_dir} 中未找到包含 {ir_token} 的 .pt 文件。")
    candidates = sorted(candidates, key=lambda f: os.path.getmtime(os.path.join(model_dir, f)), reverse=True)
    return os.path.join(model_dir, candidates[0])

def inspect_model_state_dict(state_dict):
    """检查模型state_dict的结构并尝试推断输入维度"""
    print("\n📊 检查模型state_dict结构:")
    print(f"   state_dict类型: {type(state_dict)}")
    if isinstance(state_dict, dict):
        print(f"   包含的键: {list(state_dict.keys())[:10]}")  # 只显示前10个
        
        # 尝试多种可能的键名来推断输入维度
        possible_keys = [
            'semnet.0.weight',           # 标准命名
            'module.semnet.0.weight',    # DataParallel包装
            '_orig_mod.semnet.0.weight', # 编译后的模型
            'model.semnet.0.weight',     # 嵌套在model中
            'net.semnet.0.weight',       # 嵌套在net中
            'state_dict.semnet.0.weight' # 嵌套更深
        ]
        
        for key in possible_keys:
            if key in state_dict:
                weight_shape = state_dict[key].shape
                if len(weight_shape) == 2:  # Linear层权重应该是2维
                    input_dim = weight_shape[1]
                    output_dim = weight_shape[0]
                    print(f"   ✅ 从 '{key}' 推断: 输入维度={input_dim}, 输出维度={output_dim}")
                    return input_dim
        
        # 如果还是找不到，尝试模糊搜索
        for key, value in state_dict.items():
            if 'weight' in key.lower() and value.ndim == 2:
                # 可能是linear层的权重
                input_dim = value.shape[1]
                print(f"   🔍 从 '{key}' (shape={value.shape}) 推断输入维度={input_dim}")
                return input_dim
    
    return None

# ---- Simple Dataset for future pairs (适配 v1, v2) ----
class FuturePairsDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        
        # ID 列现在明确是 'v1' 和 'v2' (来自 solution 文件)
        self.id_cols = ['v1', 'v2'] 
        
        # 特征列是除了 ID 列之外的所有数值列 (即列 '0' 到 '140')
        non_features = set(self.id_cols)
        feature_cols = [c for c in df.columns if c not in non_features and np.issubdtype(df[c].dtype, np.number)]
            
        self.feature_cols = feature_cols
        
        # ------------------- 检查和报告 -------------------
        if len(self.feature_cols) != 141:
            # 这是一个关键检查，确保特征数量是 141
            print(f"⚠️ 警告: 实际提取的特征数量为 {len(self.feature_cols)}，模型要求 141。")
        else:
            print(f"✅ 实际提取的特征数量: {len(self.feature_cols)} (列 '{self.feature_cols[0]}' 到 '{self.feature_cols[-1]}')")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # ID 直接从 int64 转换为字符串 (安全)
        concept1_id = str(row[self.id_cols[0]])
        concept2_id = str(row[self.id_cols[1]])
            
        # 提取 141 列特征 (这些是归一化的 float)
        feat = row[self.feature_cols].values.astype(np.float32)
        pair = (concept1_id, concept2_id)
        
        return feat, pair


# ---------------- Main ----------------
def main():
    # 1) 找最优 IR
    print("🔎 解析 result 文件，寻找最优模型 ...")
    best_ir, best_eval, df_results = find_best_ir(RESULT_FILE)
    print(f"✅ 找到最优 IR={best_ir} (eval={best_eval:.4f})")

    # 2) 找对应模型文件
    best_model_path = find_model_file_for_ir(MODEL_DIR, best_ir)
    print(f"✅ 对应模型文件: {best_model_path}")

    # 3) 设备
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("🖥️ 使用设备:", device)

    # 4) 模型结构参数 (保持不变)
    DEFAULT_HIDDEN_SIZE = 600
    DEFAULT_OUTPUT_SIZE = 1

    # 5) 加载 checkpoint (改进版)
    ck = torch.load(best_model_path, map_location='cpu')
    print(f"📦 Checkpoint类型: {type(ck)}")
    
    # 尝试多种方式提取state_dict
    if isinstance(ck, dict):
        # 尝试不同的常见键名
        candidate_keys = ['model_state_dict', 'state_dict', 'net', 'model', 'module']
        state_dict = None
        
        for key in candidate_keys:
            if key in ck:
                print(f"   找到 '{key}' 键，使用该字典作为state_dict")
                state_dict = ck[key]
                break
        
        if state_dict is None:
            # 如果没有找到特定的键，假设整个ck就是state_dict
            # 但需要过滤掉非张量项
            print("   未找到标准键名，尝试过滤非张量项...")
            state_dict = {k: v for k, v in ck.items() if isinstance(v, torch.Tensor)}
            if not state_dict:
                state_dict = ck  # 保持原样，让后续代码处理
    else:
        state_dict = ck
    
    # 6) 推断输入维度（改进版）
    inferred_input_size = inspect_model_state_dict(state_dict)
    
    if inferred_input_size is None:
        # 如果仍然无法推断，尝试使用特征文件来确定输入维度
        print("⚠️ 无法从模型推断输入维度，尝试从特征文件确定...")
        if os.path.exists(FEATURE_FILE):
            df_test = pd.read_parquet(FEATURE_FILE)
            inferred_input_size = df_test.shape[1]
            print(f"✅ 从特征文件推断输入维度: {inferred_input_size}")
        else:
            # 最后的手段：使用默认值141
            print("⚠️ 使用默认输入维度: 141")
            inferred_input_size = 141
    
    input_size = inferred_input_size
    print(f"📐 最终模型输入维度 = {input_size}, 隐藏层 = {DEFAULT_HIDDEN_SIZE}, 输出 = {DEFAULT_OUTPUT_SIZE}")

    # 7) 构建模型
    model = ff_network(input_size, DEFAULT_HIDDEN_SIZE, DEFAULT_OUTPUT_SIZE)
    print("✅ 使用 ff_network 作为模型结构。")

    # 8) 加载 state_dict (改进版)
    try:
        # 尝试直接加载
        model.load_state_dict(state_dict)
        print("✅ 成功加载state_dict")
    except Exception as e:
        print(f"⚠️ 直接加载失败: {e}")
        try:
            # 移除可能的 'module.' 前缀
            new_state_dict = {}
            for k, v in state_dict.items():
                new_key = k.replace('module.', '')
                new_state_dict[new_key] = v
            model.load_state_dict(new_state_dict)
            print("✅ 成功加载state_dict (移除 'module.' 前缀后)")
        except Exception as e2:
            print(f"❌ 加载失败: {e2}")
            raise RuntimeError(f"无法加载模型权重: {e2}")

    model.to(device)
    model.eval()
    print("✅ 模型已加载并设置为评估模式。")

    # 9) 加载 ID 和特征文件
    print("\n📂 加载数据文件...")
    
    # 9.1) 加载特征文件 (141 列归一化特征)
    if not os.path.exists(FEATURE_FILE):
        raise FileNotFoundError(f"未找到特征文件: {FEATURE_FILE}")
    df_features = pd.read_parquet(FEATURE_FILE)
    print(f"✅ 特征文件加载: {df_features.shape}")
    
    # 9.2) 加载 ID 文件 (包含 v1, v2)
    if not os.path.exists(ID_FILE):
        raise FileNotFoundError(f"未找到 ID 文件: {ID_FILE}")
    df_ids = pd.read_parquet(ID_FILE)[['v1', 'v2']]  # 只提取 v1 和 v2
    print(f"✅ ID文件加载: {df_ids.shape}")
    
    # 9.3) 合并：确保特征和 ID 数量一致
    if len(df_features) != len(df_ids):
        raise ValueError(f"特征文件 ({len(df_features)} 行) 和 ID 文件 ({len(df_ids)} 行) 行数不匹配！无法预测。")
        
    # 重置索引以便合并，并确保列名不冲突
    df_ids = df_ids.reset_index(drop=True)
    df_features = df_features.reset_index(drop=True)

    # 最终的预测数据：ID + 141 特征
    df_future = pd.concat([df_ids, df_features], axis=1)
    
    del df_ids, df_features
    gc.collect()

    print(f"✅ 最终预测数据集: {df_future.shape[0]} 行, {df_future.shape[1]} 列")
    print(f"   列名示例: {df_future.columns[:5].tolist()} ...")

    # 10) 预测
    print("\n🔮 开始预测...")
    ds = FuturePairsDataset(df_future)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
    scores, pairs = [], []

    start_t = time.time()
    with torch.no_grad():
        for i, (x_batch, pair_batch) in enumerate(loader):
            x = x_batch.to(device)

            # 验证输入维度匹配
            expected_input_dim = model.semnet[0].in_features
            current_input_dim = x.shape[1]
            
            if current_input_dim != expected_input_dim:
                print(f"⚠️ 批次 {i}: 输入维度不匹配! 期望 {expected_input_dim}, 实际 {current_input_dim}")
                if current_input_dim < expected_input_dim:
                    # 补齐特征
                    padding = torch.zeros(x.shape[0], expected_input_dim - current_input_dim).to(device)
                    x = torch.cat([x, padding], dim=1)
                    print(f"   已补齐到 {expected_input_dim} 维")
                else:
                    # 截断特征
                    x = x[:, :expected_input_dim]
                    print(f"   已截断到 {expected_input_dim} 维")
            
            out = model(x)
            if out.ndim == 2 and out.shape[1] == 1:
                probs = torch.sigmoid(out).squeeze(dim=1)
            elif out.ndim == 1:
                probs = torch.sigmoid(out)
            elif out.ndim == 2 and out.shape[1] == 2:
                probs = F.softmax(out, dim=1)[:, 1]
            else:
                probs = torch.sigmoid(out).squeeze()

            scores.extend(probs.cpu().numpy().tolist())
            
            c1_ids = pair_batch[0]
            c2_ids = pair_batch[1]
            pairs.extend(list(zip(c1_ids, c2_ids)))
            
            if (i+1) % 10 == 0:
                print(f"   已处理 {i+1} 批次, 用时 {time.time()-start_t:.1f}s")

    # 11) 导出结果
    print("\n💾 保存预测结果...")
    df_out = pd.DataFrame(pairs, columns=['concept1', 'concept2'])
    df_out['score'] = scores
    df_out = df_out.sort_values('score', ascending=False).reset_index(drop=True)
    out_csv = os.path.join(OUTPUT_DIR, "future_concept_pairs_sorted.csv")
    df_out.to_csv(out_csv, index=False, encoding='utf-8-sig')
    print(f"✅ 结果已保存: {out_csv}")
    print(f"📊 预测完成! 共 {len(df_out)} 对概念")
    print("\n🏆 Top 10 预测结果:")
    print(df_out.head(10))
    
    # 显示统计信息
    print(f"\n📈 预测分数统计:")
    print(f"   最小值: {df_out['score'].min():.4f}")
    print(f"   最大值: {df_out['score'].max():.4f}")
    print(f"   平均值: {df_out['score'].mean():.4f}")
    print(f"   中位数: {df_out['score'].median():.4f}")


if __name__ == "__main__":
    main()

In [ ]:
# run_best_predict_with_transformer.py (使用正确的Transformer模型)
import os
import re
import time
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn 
import gc
import math

# ---------------- Config ----------------
BASE_DIR = r"E:\study\research\impact4cast_cscl"
RESULT_FILE = os.path.join(BASE_DIR, "2019_train", "t0_c2_result", "All_AUC_Report_Year_Train_2019.txt")
MODEL_DIR = os.path.join(BASE_DIR, "2019_train", "t0_c2_net")
OUTPUT_DIR = os.path.join(BASE_DIR, "predictions")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 真正的预测特征文件 (141 列归一化特征)
FEATURE_FILE = os.path.join(BASE_DIR, "data_eval", "eval_data_pair_feature.parquet")

# 包含原始 ID 和可能标签的解决方案文件 (包含 v1, v2)
ID_FILE = os.path.join(BASE_DIR, "data_eval", "data_eval_pair_solution.parquet") 

BATCH_SIZE = 2048


# ---------------- Transformer模型定义 (根据您的checkpoint结构) ----------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

class FeatureProjection(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(FeatureProjection, self).__init__()
        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.projection(x)

class FeatureAggregator(nn.Module):
    def __init__(self, d_model, nhead=8):
        super(FeatureAggregator, self).__init__()
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, query, key_value):
        attn_output, _ = self.cross_attn(query, key_value, key_value)
        return self.norm(query + attn_output)

class OutputLayer(nn.Module):
    def __init__(self, d_model, output_dim=1):
        super(OutputLayer, self).__init__()
        self.output_net = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, d_model // 4),
            nn.ReLU(),
            nn.Linear(d_model // 4, output_dim)
        )
    
    def forward(self, x):
        return self.output_net(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim=141, d_model=256, nhead=8, num_layers=4, output_dim=1):
        super(TransformerModel, self).__init__()
        
        # 特征投影
        self.feature_proj = FeatureProjection(input_dim, d_model)
        
        # 位置编码
        self.pos_encoder = PositionalEncoding(d_model)
        
        # Transformer编码器层
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=d_model * 4,
            dropout=0.1,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 特征聚合器
        self.feature_aggregator = FeatureAggregator(d_model, nhead)
        
        # 输出层
        self.output_layer = OutputLayer(d_model, output_dim)
        
    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        # 如果输入是2维，添加序列维度
        if x.dim() == 2:
            x = x.unsqueeze(1)  # (batch_size, 1, input_dim)
        
        # 特征投影
        x = self.feature_proj(x)  # (batch_size, seq_len, d_model)
        
        # 位置编码
        x = self.pos_encoder(x)
        
        # Transformer编码
        x = self.transformer_encoder(x)
        
        # 特征聚合（使用CLS token或平均池化）
        # 这里使用第一个token作为聚合特征
        aggregated = self.feature_aggregator(x[:, 0:1, :], x)
        
        # 输出
        output = self.output_layer(aggregated.squeeze(1))
        
        return output

# ---------------- helper functions ----------------
def find_best_ir(result_file_path):
    pattern = re.compile(r'IR=\[(\d+)\]: train=([\d.]+); test=([\d.]+); eval=([\d.]+)')
    results = []
    with open(result_file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            m = pattern.search(line)
            if m:
                ir, train, test, eval_ = m.groups()
                results.append({
                    'IR': int(ir),
                    'train': float(train),
                    'test': float(test),
                    'eval': float(eval_)
                })
    if not results:
        raise RuntimeError(f"未在 {result_file_path} 中解析到任何 IR 记录")
    df = pd.DataFrame(results)
    best_ir = int(df.loc[df['eval'].idxmax(), 'IR'])
    best_eval = float(df['eval'].max())
    return best_ir, best_eval, df

def find_model_file_for_ir(model_dir, ir):
    ir_token = f"IR_{ir:03d}"
    candidates = [f for f in os.listdir(model_dir) if ir_token in f and f.endswith('.pt')]
    if not candidates:
        ir_token2 = f"IR_{ir}"
        candidates = [f for f in os.listdir(model_dir) if ir_token2 in f and f.endswith('.pt')]
    if not candidates:
        raise FileNotFoundError(f"在 {model_dir} 中未找到包含 {ir_token} 的 .pt 文件")
    candidates = sorted(candidates, key=lambda f: os.path.getmtime(os.path.join(model_dir, f)), reverse=True)
    return os.path.join(model_dir, candidates[0])

# ---------------- Dataset ----------------
class FuturePairsDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.id_cols = ['v1', 'v2']
        
        # 特征列
        non_features = set(self.id_cols)
        feature_cols = [c for c in df.columns if c not in non_features and np.issubdtype(df[c].dtype, np.number)]
        self.feature_cols = feature_cols
        
        if len(self.feature_cols) != 141:
            print(f"⚠️ 警告: 实际特征数量为 {len(self.feature_cols)}，期望 141")
        else:
            print(f"✅ 特征数量: {len(self.feature_cols)}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        concept1_id = str(row[self.id_cols[0]])
        concept2_id = str(row[self.id_cols[1]])
        feat = row[self.feature_cols].values.astype(np.float32)
        pair = (concept1_id, concept2_id)
        return feat, pair

# ---------------- Main ----------------
def main():
    # 1) 找最优 IR
    print("🔎 解析 result 文件，寻找最优模型 ...")
    best_ir, best_eval, df_results = find_best_ir(RESULT_FILE)
    print(f"✅ 找到最优 IR={best_ir} (eval={best_eval:.4f})")

    # 2) 找对应模型文件
    best_model_path = find_model_file_for_ir(MODEL_DIR, best_ir)
    print(f"✅ 对应模型文件: {best_model_path}")

    # 3) 设备
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🖥️ 使用设备: {device}")

    # 4) 加载checkpoint
    print("📦 加载模型权重...")
    checkpoint = torch.load(best_model_path, map_location='cpu')
    
    # 检查checkpoint格式
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
        print("   从 checkpoint 中提取 model_state_dict")
    elif isinstance(checkpoint, dict) and any('transformer_encoder' in k for k in checkpoint.keys()):
        state_dict = checkpoint
        print("   直接使用 checkpoint 作为 state_dict")
    elif isinstance(checkpoint, nn.Module):
        state_dict = checkpoint.state_dict()
        print("   checkpoint 是 nn.Module，提取 state_dict")
    else:
        state_dict = checkpoint
        print(f"   checkpoint 类型: {type(checkpoint)}")
    
    # 5) 创建模型
    print("🏗️ 构建 Transformer 模型...")
    # 从state_dict推断参数
    d_model = None
    if 'feature_proj.projection.0.weight' in state_dict:
        d_model = state_dict['feature_proj.projection.0.weight'].shape[0]
    
    num_layers = 0
    for key in state_dict.keys():
        if 'transformer_encoder.layers.' in key:
            layer_num = int(key.split('.')[2])
            num_layers = max(num_layers, layer_num + 1)
    
    print(f"   推断参数: d_model={d_model}, num_layers={num_layers}")
    
    # 创建模型
    model = TransformerModel(
        input_dim=141,
        d_model=d_model or 256,
        nhead=8,
        num_layers=num_layers or 4,
        output_dim=1
    )
    
    # 6) 加载权重
    try:
        # 移除可能的 'module.' 前缀
        new_state_dict = {}
        for k, v in state_dict.items():
            new_key = k.replace('module.', '')
            new_state_dict[new_key] = v
        
        # 加载权重
        missing_keys, unexpected_keys = model.load_state_dict(new_state_dict, strict=False)
        
        if missing_keys:
            print(f"⚠️ 缺失的键: {missing_keys[:5]}...")
        if unexpected_keys:
            print(f"⚠️ 额外的键: {unexpected_keys[:5]}...")
        
        print("✅ 模型权重加载成功")
    except Exception as e:
        print(f"❌ 加载失败: {e}")
        raise
    
    model.to(device)
    model.eval()
    
    # 7) 加载数据
    print("\n📂 加载数据文件...")
    
    if not os.path.exists(FEATURE_FILE):
        raise FileNotFoundError(f"未找到特征文件: {FEATURE_FILE}")
    df_features = pd.read_parquet(FEATURE_FILE)
    print(f"✅ 特征文件: {df_features.shape}")
    
    if not os.path.exists(ID_FILE):
        raise FileNotFoundError(f"未找到 ID 文件: {ID_FILE}")
    df_ids = pd.read_parquet(ID_FILE)[['v1', 'v2']]
    print(f"✅ ID文件: {df_ids.shape}")
    
    if len(df_features) != len(df_ids):
        raise ValueError(f"行数不匹配: 特征 {len(df_features)} vs ID {len(df_ids)}")
    
    df_ids = df_ids.reset_index(drop=True)
    df_features = df_features.reset_index(drop=True)
    df_future = pd.concat([df_ids, df_features], axis=1)
    
    del df_ids, df_features
    gc.collect()
    
    print(f"✅ 最终数据集: {len(df_future)} 行")
    
    # 8) 预测
    print("\n🔮 开始预测...")
    ds = FuturePairsDataset(df_future)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
    scores, pairs = [], []
    
    start_t = time.time()
    with torch.no_grad():
        for i, (x_batch, pair_batch) in enumerate(loader):
            x = x_batch.to(device)
            
            # 模型前向传播
            out = model(x)
            
            # 转换为概率
            if out.ndim == 2 and out.shape[1] == 1:
                probs = torch.sigmoid(out).squeeze(dim=1)
            elif out.ndim == 1:
                probs = torch.sigmoid(out)
            elif out.ndim == 2 and out.shape[1] == 2:
                probs = F.softmax(out, dim=1)[:, 1]
            else:
                probs = torch.sigmoid(out).squeeze()
            
            scores.extend(probs.cpu().numpy().tolist())
            
            c1_ids = pair_batch[0]
            c2_ids = pair_batch[1]
            pairs.extend(list(zip(c1_ids, c2_ids)))
            
            if (i+1) % 10 == 0:
                print(f"   已处理 {i+1} 批次 ({len(scores)} 样本), 用时 {time.time()-start_t:.1f}s")
    
    # 9) 保存结果
    print("\n💾 保存预测结果...")
    df_out = pd.DataFrame(pairs, columns=['concept1', 'concept2'])
    df_out['score'] = scores
    df_out = df_out.sort_values('score', ascending=False).reset_index(drop=True)
    out_csv = os.path.join(OUTPUT_DIR, "future_concept_pairs_sorted.csv")
    df_out.to_csv(out_csv, index=False, encoding='utf-8-sig')
    
    print(f"✅ 结果已保存: {out_csv}")
    print(f"📊 共预测 {len(df_out)} 对概念")
    print("\n🏆 Top 20 预测结果:")
    print(df_out.head(20))
    
    print(f"\n📈 预测分数统计:")
    print(f"   最小值: {df_out['score'].min():.6f}")
    print(f"   最大值: {df_out['score'].max():.6f}")
    print(f"   平均值: {df_out['score'].mean():.6f}")
    print(f"   中位数: {df_out['score'].median():.6f}")
    
    # 显示分数分布
    print(f"\n📊 分数分布:")
    print(f"   >0.9: {(df_out['score'] > 0.9).sum()}")
    print(f"   0.8-0.9: {((df_out['score'] > 0.8) & (df_out['score'] <= 0.9)).sum()}")
    print(f"   0.7-0.8: {((df_out['score'] > 0.7) & (df_out['score'] <= 0.8)).sum()}")
    print(f"   0.6-0.7: {((df_out['score'] > 0.6) & (df_out['score'] <= 0.7)).sum()}")
    print(f"   <=0.6: {(df_out['score'] <= 0.6).sum()}")


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import os

# ---------------- 配置 ----------------
BASE_DIR = r"E:\study\research\impact4cast_cscl"
# 预测结果文件路径
INPUT_CSV_PATH = os.path.join(BASE_DIR, "predictions", "future_concept_pairs_sorted.csv")
# 概念关键词文件路径
CONCEPT_FILE_PATH = r"E:\study\research\ASIST\entities.txt"
# 输出文件路径
OUTPUT_CSV_PATH = os.path.join(BASE_DIR, "predictions", "top10_with_keywords.csv")

# Ensure the file exists before attempting to read it
if os.path.exists(INPUT_CSV_PATH):
    print(f"Reading the first 10 lines of the file: {INPUT_CSV_PATH}")
    with open(INPUT_CSV_PATH, 'r', encoding='utf-8') as file:
        for i in range(10):
            print(file.readline().strip())
else:
    print(f"❌ File not found: {INPUT_CSV_PATH}")

In [ ]:
import pandas as pd
import os

# ---------------- 配置 ----------------
BASE_DIR = r"E:\study\research\impact4cast_cscl"
# 预测结果文件路径
INPUT_CSV_PATH = os.path.join(BASE_DIR, "predictions", "future_concept_pairs_sorted.csv")
# 概念关键词文件路径
CONCEPT_FILE_PATH =r"E:\study\research\ASIST\entities.txt"
# 输出文件路径
OUTPUT_CSV_PATH = os.path.join(BASE_DIR, "predictions", "top10_with_keywords.csv")

# 你的 Top 10 数据（直接用于演示，实际运行会从 INPUT_CSV_PATH 读取）
top_10_data = {
    'concept1': [4777.0, 19776.0, 15794.0, 11133.0, 12311.0, 17840.0, 18947.0, 226.0, 6431.0, 27432.0],
    'concept2': [5597.0, 29060.0, 35278.0, 35171.0, 24526.0, 35309.0, 23462.0, 5383.0, 23701.0, 29417.0],
    'score': [0.750425, 0.747279, 0.746214, 0.746155, 0.745913, 0.745905, 0.745553, 0.745519, 0.744880, 0.744153]
}

# ---------------- 函数 ----------------

def load_concept_map(file_path):
    """
    加载 full_domain_concepts.txt 文件，创建 ID (索引) 到关键词的映射。
    假定文件每行一个关键词，索引从 0 开始。
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ 关键词文件不存在：{file_path}")

    print(f"✅ 正在加载关键词文件: {file_path}")
    concept_map = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        # 文件中的行号 i 即为概念 ID
        for i, line in enumerate(f):
            keyword = line.strip()
            # 概念 ID 以整数形式存储
            concept_map[i] = keyword
            
    print(f"✅ 已加载 {len(concept_map)} 个关键词。")
    return concept_map

def process_top_results(input_df, concept_map, output_path):
    """
    处理 Top 10 结果，查找关键词并保存到 CSV。
    """
    # 确保 ID 是整数以便查找，因为 concept_map 的键是整数。
    df = input_df.copy()
    df['concept1_id'] = df['concept1'].astype(int)
    df['concept2_id'] = df['concept2'].astype(int)
    
    # 使用 map 函数进行关键词查找
    # 注意：如果 ID 不在 concept_map 中，map 会返回 NaN
    df['concept1_keyword'] = df['concept1_id'].map(concept_map).fillna('Keyword Not Found')
    df['concept2_keyword'] = df['concept2_id'].map(concept_map).fillna('Keyword Not Found')

    # 重新排列和选择最终列
    final_df = df[['concept1_id', 'concept1_keyword', 'concept2_id', 'concept2_keyword', 'score']]
    
    # 保存结果
    final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 成功生成带关键词的 CSV 文件：{output_path}")
    print("\n--- Top 10 关键词结果预览 ---")
    print(final_df.head(10).to_markdown(index=False))
    
    return final_df

# ---------------- 主程序 ----------------
try:
    # 1. 加载关键词映射
    concept_map = load_concept_map(CONCEPT_FILE_PATH)
    
    # 2. 从之前生成的 CSV 文件中读取完整的 Top N 结果（如果文件存在）
    if os.path.exists(INPUT_CSV_PATH):
        print(f"正在读取预测结果文件: {INPUT_CSV_PATH}")
        df_results = pd.read_csv(INPUT_CSV_PATH).head(10) # 只取前 10 行
    else:
        # 如果文件不存在，使用你提供的 Top 10 示例数据进行演示
        print(f"⚠️ 预测结果 CSV 文件不存在 ({INPUT_CSV_PATH})，使用示例数据进行演示。")
        df_results = pd.DataFrame(top_10_data)

    # 3. 处理并生成带关键词的 CSV
    process_top_results(df_results, concept_map, OUTPUT_CSV_PATH)

except FileNotFoundError as e:
    print(f"\n❌ 文件错误：{e}")
except Exception as e:
    print(f"\n❌ 发生意外错误：{e}")

# ---------------- 下一步 ----------------
print("\n--- 下一步：请打开生成的 CSV 文件查看完整结果 ---")

In [ ]:
import pandas as pd
import os

# ---------------- 配置 ----------------
BASE_DIR = r"E:\study\research\impact4cast_cscl"

# 1. 预测结果文件路径 (输入)
INPUT_CSV_PATH = os.path.join(BASE_DIR, "predictions", "future_concept_pairs_sorted.csv")

# 2. 概念关键词文件路径 (映射表)
CONCEPT_FILE_PATH = r"E:\study\research\ASIST\entities.txt"

# 3. 输出文件路径 (输出：包含所有结果)
OUTPUT_CSV_PATH = os.path.join(BASE_DIR, "predictions", "all_predictions_with_entities.csv")

# ---------------- 函数 ----------------

def load_concept_map(file_path):
    """
    加载 full_domain_concepts.txt 文件，创建 ID (索引) 到关键词的映射。
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ 关键词文件不存在：{file_path}")

    print(f"⏳ 正在加载关键词文件: {file_path}")
    concept_map = {}
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            # 文件中的行号 i 即为概念 ID
            for i, line in enumerate(f):
                keyword = line.strip()
                if keyword: # 避免空行
                    concept_map[i] = keyword
        print(f"✅ 已加载 {len(concept_map)} 个关键词映射。")
        return concept_map
    except Exception as e:
        raise Exception(f"读取关键词文件失败: {e}")

def process_all_results(input_path, concept_map, output_path):
    """
    读取完整的预测文件，匹配关键词并保存。
    """
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"❌ 找不到预测结果文件：{input_path}")

    print(f"⏳ 正在读取预测结果 CSV (这可能需要一点时间)...")
    # 读取所有数据，不再使用 .head(10)
    df = pd.read_csv(input_path)
    print(f"✅ 读取成功，共 {len(df)} 条数据。")

    print("⏳ 正在进行关键词匹配...")
    # 1. 确保 ID 列是整数类型 (CSV中可能是 4777.0 这种浮点数)
    # 使用 fillna(0) 防止空值报错，但理论上不应有空值
    df['concept1_id'] = df['concept1'].fillna(-1).astype(int)
    df['concept2_id'] = df['concept2'].fillna(-1).astype(int)
    
    # 2. 使用 map 函数进行大规模关键词匹配
    # 如果 ID 在 map 中找不到，会显示 'Unknown ID'
    df['concept1_keyword'] = df['concept1_id'].map(concept_map).fillna('Unknown ID')
    df['concept2_keyword'] = df['concept2_id'].map(concept_map).fillna('Unknown ID')

    # 3. 整理列顺序
    final_df = df[['concept1_id', 'concept1_keyword', 'concept2_id', 'concept2_keyword', 'score']]
    
    print(f"⏳ 正在保存结果到: {output_path}")
    # 保存结果
    final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"✅ 保存完成！")
    
    return final_df

# ---------------- 主程序 ----------------
if __name__ == "__main__":
    try:
        # 1. 加载关键词映射
        concept_map = load_concept_map(CONCEPT_FILE_PATH)
        
        # 2. 处理所有数据
        process_all_results(INPUT_CSV_PATH, concept_map, OUTPUT_CSV_PATH)

        print("\n" + "="*50)
        print(f"处理完毕。所有关键词的分数结果已保存至：\n{OUTPUT_CSV_PATH}")
        print("="*50)

    except FileNotFoundError as e:
        print(f"\n❌ 文件路径错误：{e}")
        print("请检查 BASE_DIR 路径是否正确。")
    except Exception as e:
        print(f"\n❌ 程序运行出错：{e}")

In [ ]:
# generate_train_data_2022_2025.py
import os
import re
import time
import torch
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pickle
import gzip
import gc
from datetime import datetime, date
import warnings
import torch.nn as nn  
warnings.filterwarnings('ignore')

# ---------------- Config ----------------
BASE_DIR = r"E:\study\research\impact4cast_cscl"
RESULT_FILE = os.path.join(BASE_DIR, "2019_train", "t0_c2_result", "All_AUC_Report_Year_Train_2019.txt")
MODEL_DIR = os.path.join(BASE_DIR, "2019_train", "t0_c2_net")
OUTPUT_DIR = os.path.join(BASE_DIR, "data_for_train_2025")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 数据文件路径
FEATURE_FOLDER = os.path.join(BASE_DIR, "data_for_features")
DYNAMIC_GRAPH_FILE = os.path.join(BASE_DIR, "data_concept_graph", "full_dynamic_graph.parquet")
CONCEPT_LIST_FILE = r"E:\study\research\ASIST\entities.txt" # 概念列表文件

# 新增：特征文件路径（包含141维归一化特征）
FEATURE_FILE = os.path.join(BASE_DIR, "data_eval", "eval_data_pair_feature.parquet")
ID_FILE = os.path.join(BASE_DIR, "data_eval", "data_eval_pair_solution.parquet")

# 时间设置
YEAR_START = 2022  # 特征基准年
YEAR_END = 2025    # 标签年
YEARS_DELTA = 3    # 时间窗口

# 分批处理参数
BATCH_SIZE = 1000000  # 每批处理的概念对数量
MAX_PAIRS = None    # 先设置为1000进行测试，完成后可以设为None处理所有

# ---------------- 模型定义（用于特征提取，如果需要） ----------------
class ff_network(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ff_network, self).__init__()
        self.semnet = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size)
        )

    def forward(self, x):
        return self.semnet(x)


# ---------------- 辅助函数 ----------------
def find_best_ir(result_file_path):
    """从结果文件中找到最优IR值（基于eval AUC）"""
    pattern = re.compile(r'IR=\[(\d+)\]: train=([\d.]+); test=([\d.]+); eval=([\d.]+)')
    results = []
    
    print(f"🔎 解析结果文件: {result_file_path}")
    with open(result_file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            m = pattern.search(line)
            if m:
                ir, train, test, eval_ = m.groups()
                results.append({
                    'IR': int(ir),
                    'train': float(train),
                    'test': float(test),
                    'eval': float(eval_)
                })
    
    if not results:
        raise RuntimeError(f"未在 {result_file_path} 中解析到任何IR记录，请检查文件格式。")
    
    df = pd.DataFrame(results)
    best_idx = df['eval'].idxmax()
    best_ir = int(df.loc[best_idx, 'IR'])
    best_eval = float(df.loc[best_idx, 'eval'])
    
    print(f"✅ 找到最优IR={best_ir} (eval AUC={best_eval:.4f})")
    print("\n所有IR的AUC分数:")
    print(df.to_string(index=False))
    
    return best_ir, best_eval, df


def load_feature_mapping(feature_file, id_file):
    """
    加载特征文件和ID文件，创建概念对到特征的映射
    
    返回:
        feature_map: 字典，键为(v1, v2)元组，值为141维特征数组
    """
    print(f"🔎 加载特征映射文件...")
    
    # 加载特征文件
    if not os.path.exists(feature_file):
        raise FileNotFoundError(f"特征文件不存在: {feature_file}")
    
    print(f"加载特征文件: {feature_file}")
    df_features = pd.read_parquet(feature_file)
    print(f"特征文件形状: {df_features.shape}")
    
    # 加载ID文件
    if not os.path.exists(id_file):
        raise FileNotFoundError(f"ID文件不存在: {id_file}")
    
    print(f"加载ID文件: {id_file}")
    df_ids = pd.read_parquet(id_file)
    
    # 确保只取需要的列
    if 'v1' in df_ids.columns and 'v2' in df_ids.columns:
        df_ids = df_ids[['v1', 'v2']]
    else:
        # 如果列名不是v1, v2，假设前两列是ID
        df_ids = df_ids.iloc[:, :2]
        df_ids.columns = ['v1', 'v2']
    
    print(f"ID文件形状: {df_ids.shape}")
    
    # 检查行数是否匹配
    if len(df_features) != len(df_ids):
        raise ValueError(f"特征文件({len(df_features)}行)和ID文件({len(df_ids)}行)行数不匹配！")
    
    # 合并ID和特征
    df_combined = pd.concat([df_ids.reset_index(drop=True), 
                             df_features.reset_index(drop=True)], axis=1)
    
    # 创建映射字典
    feature_map = {}
    
    print("创建概念对到特征的映射...")
    for idx, row in df_combined.iterrows():
        v1, v2 = int(row['v1']), int(row['v2'])
        # 确保v1 < v2以统一表示
        if v1 > v2:
            v1, v2 = v2, v1
        
        # 提取特征（从第3列开始）
        features = row.iloc[2:].values.astype(np.float32)
        
        # 验证特征维度
        if len(features) != 141:
            print(f"⚠️ 警告: 特征维度为 {len(features)}，应为141")
        
        feature_map[(v1, v2)] = features
    
    print(f"✅ 成功创建 {len(feature_map)} 个概念对的映射")
    
    # 显示特征统计
    if len(feature_map) > 0:
        sample_features = next(iter(feature_map.values()))
        print(f"特征维度: {len(sample_features)}")
        print(f"特征范围: [{sample_features.min():.4f}, {sample_features.max():.4f}]")
        print(f"特征均值: {sample_features.mean():.4f}")
    
    return feature_map


def get_unconnected_pairs_batch(graph_file, year_start, batch_size=100000, max_pairs=None):
    """
    分批获取在year_start年未连接的概念对
    
    返回: 生成器，每批返回 (batch_pairs, total_count)
    """
    print(f"🔎 正在查找 {year_start}年未连接的概念对...")
    
    # 1. 获取截至year_start的所有边
    pf = pq.ParquetFile(graph_file)
    
    # 先收集所有存在的边
    existing_pairs = set()
    total_edges = 0
    
    print("正在收集已存在的概念对...")
    for batch in pf.iter_batches(batch_size=batch_size):
        df_batch = batch.to_pandas()
        
        # 检查是否有time列
        time_col = None
        for col in df_batch.columns:
            if 'time' in col.lower() or 'date' in col.lower() or 'year' in col.lower():
                time_col = col
                break
        
        # 筛选时间 <= year_start的边
        if time_col is not None:
            try:
                # 如果time_col是整数类型（如年份）
                if pd.api.types.is_integer_dtype(df_batch[time_col]):
                    df_batch = df_batch[df_batch[time_col] <= year_start]
                # 如果time_col是字符串类型
                elif pd.api.types.is_string_dtype(df_batch[time_col]):
                    year_start_str = str(year_start)
                    df_batch = df_batch[df_batch[time_col].str.startswith(year_start_str) | 
                                         (df_batch[time_col] < str(year_start + 1))]
                # 如果time_col是日期类型
                else:
                    year_start_date = pd.Timestamp(f"{year_start}-12-31")
                    df_batch = df_batch[pd.to_datetime(df_batch[time_col]) <= year_start_date]
            except Exception as e:
                print(f"  时间筛选时出错: {e}，跳过时间筛选")
        
        # 添加边到集合
        for _, row in df_batch.iterrows():
            v1, v2 = row['v1'], row['v2']
            # 确保v1 < v2以统一表示
            if v1 < v2:
                existing_pairs.add((v1, v2))
            else:
                existing_pairs.add((v2, v1))
        
        total_edges += len(df_batch)
        if len(existing_pairs) % 100000 == 0:
            print(f"  已收集 {len(existing_pairs)} 个唯一概念对")
    
    print(f"✅ 共收集到 {len(existing_pairs)} 个已存在的概念对（截至{year_start}年）")
    
    # 2. 获取所有概念ID
    # 从邻接矩阵或概念列表获取
    concepts = set()
    for year in [year_start, year_start-1, year_start-2]:
        cite_file = os.path.join(FEATURE_FOLDER, f"concept_node_citation_data_{year}.parquet")
        if os.path.exists(cite_file):
            try:
                df_cite = pd.read_parquet(cite_file)
                # 假设第一列是概念ID
                concepts.update(df_cite.iloc[:, 0].tolist())
            except Exception as e:
                print(f"  加载 {year}年概念数据失败: {e}")
    
    concepts = list(concepts)
    if len(concepts) == 0:
        # 如果没有找到概念，从现有的边中提取
        for v1, v2 in existing_pairs:
            concepts.add(v1)
            concepts.add(v2)
        concepts = list(concepts)
    
    print(f"✅ 共 {len(concepts)} 个概念")
    
    # 3. 生成所有可能的概念对
    from itertools import combinations
    
    total_possible = len(concepts) * (len(concepts) - 1) // 2
    print(f"📊 所有可能的概念对总数: {total_possible:,}")
    
    # 限制最大处理数量
    #max_to_process = max_pairs if max_pairs else min(total_possible, 1000000)  # 默认最多处理100万
    #print(f"📊 将处理最多 {max_to_process:,} 个概念对")
    
    # 分批生成未连接的概念对
    unconnected_batch = []
    batch_count = 0
    total_unconnected = 0
    
    for i, (v1, v2) in enumerate(combinations(concepts, 2)):
        # 确保v1 < v2
        if v1 > v2:
            v1, v2 = v2, v1
        
        # 检查是否已存在
        if (v1, v2) not in existing_pairs:
            unconnected_batch.append((v1, v2))
            total_unconnected += 1
        
        # 每收集够batch_size或达到最大限制，就yield一批
        if len(unconnected_batch) >= batch_size or (max_pairs and total_unconnected >= max_pairs):
            if unconnected_batch:  # 确保批次不为空
                batch_count += 1
                print(f"  生成批次 {batch_count}: {len(unconnected_batch)} 个未连接概念对")
                yield unconnected_batch, total_unconnected
                unconnected_batch = []
        
        # 如果达到最大限制，停止
        if max_pairs and total_unconnected >= max_pairs:
            break
        
        # 每处理10万个组合打印进度
        if (i + 1) % 100000 == 0:
            print(f"  已处理 {i+1:,} 个组合，找到 {total_unconnected:,} 个未连接对")
    
    # 最后一批
    if unconnected_batch:
        batch_count += 1
        print(f"  生成批次 {batch_count}: {len(unconnected_batch)} 个未连接概念对")
        yield unconnected_batch, total_unconnected
    
    print(f"✅ 总共找到 {total_unconnected:,} 个未连接概念对")


def get_label_for_pair(v1, v2, graph_file, year_end, IR):
    """
    获取概念对在year_end年的标签
    
    如果该概念对在year_end年的引用数 ≥ IR，则标签=1，否则=0
    """
    try:
        pf = pq.ParquetFile(graph_file)
    except Exception as e:
        print(f"  无法打开图文件: {e}")
        return 0, 0
    
    total_citations = 0
    
    # 查找包含该概念对的论文
    for batch in pf.iter_batches(batch_size=100000):
        try:
            df_batch = batch.to_pandas()
        except:
            continue
        
        # 筛选包含该概念对的论文
        mask = ((df_batch['v1'] == v1) & (df_batch['v2'] == v2)) | \
               ((df_batch['v1'] == v2) & (df_batch['v2'] == v1))
        
        if mask.any():
            df_pair = df_batch[mask]
            
            # 查找引用数列
            cite_col = None
            for col in df_pair.columns:
                if col.lower().startswith('c') and col[1:].isdigit():
                    year = int(col[1:])
                    if year <= year_end:
                        cite_col = col
                        break
                elif col.lower() in ['ct', 'citation', 'citations']:
                    cite_col = col
            
            if cite_col is not None:
                total_citations += df_pair[cite_col].sum()
    
    label = 1 if total_citations >= IR else 0
    return label, total_citations


def extract_features_from_mapping(v1, v2, feature_map):
    """
    从特征映射中提取概念对(v1, v2)的141维特征
    
    参数:
        v1, v2: 概念ID
        feature_map: 特征映射字典，键为(v1, v2)元组，值为141维特征数组
    
    返回:
        141维特征列表，如果找不到则返回None
    """
    # 确保v1 < v2以统一表示
    if v1 > v2:
        v1, v2 = v2, v1
    
    # 在映射中查找
    if (v1, v2) in feature_map:
        features = feature_map[(v1, v2)]
        return features.tolist() if isinstance(features, np.ndarray) else features
    else:
        return None


# ---------------- Main ----------------
def main():
    start_time_total = time.time()
    
    # 1) 找最优IR
    print("=" * 60)
    print("步骤1: 寻找最优IR值")
    print("=" * 60)
    best_ir, best_eval, df_results = find_best_ir(RESULT_FILE)
    
    # 2) 加载特征映射
    print("\n" + "=" * 60)
    print("步骤2: 加载特征映射")
    print("=" * 60)
    feature_map = load_feature_mapping(FEATURE_FILE, ID_FILE)
    
    # 3) 准备输出文件
    output_file = os.path.join(OUTPUT_DIR, f"train_data_{YEAR_START}_{YEAR_END}_IR{best_ir}.parquet")
    temp_dir = os.path.join(OUTPUT_DIR, "temp")
    os.makedirs(temp_dir, exist_ok=True)
    
    print("\n" + "=" * 60)
    print(f"步骤3: 生成训练数据文件")
    print(f"输出文件: {output_file}")
    print(f"特征基准年: {YEAR_START}")
    print(f"标签年: {YEAR_END}")
    print(f"IR阈值: {best_ir}")
    print("=" * 60)
    
    # 4) 分批处理未连接的概念对
    print("\n📊 开始分批处理未连接概念对...")
    
    batch_files = []
    total_processed = 0
    batch_num = 0
    total_found_in_map = 0
    total_not_found = 0
    
    # 获取未连接概念对的生成器
    pair_generator = get_unconnected_pairs_batch(
        DYNAMIC_GRAPH_FILE, 
        YEAR_START, 
        batch_size=BATCH_SIZE,
        max_pairs=MAX_PAIRS
    )
    
    for batch_pairs, total_found in pair_generator:
        batch_num += 1
        batch_start = time.time()
        
        print(f"\n--- 处理批次 {batch_num} ---")
        print(f"本批概念对数量: {len(batch_pairs)}")
        print(f"累计找到概念对: {total_found}")
        
        # 处理本批概念对
        batch_data = []
        batch_found = 0
        batch_not_found = 0
        
        for i, (v1, v2) in enumerate(batch_pairs):
            if (i + 1) % 100 == 0:  # 每100个打印一次进度
                print(f"  处理本批第 {i+1}/{len(batch_pairs)} 个概念对")
            
            # 从特征映射中提取特征
            features = extract_features_from_mapping(v1, v2, feature_map)
            
            if features is not None:
                # 获取标签
                label, citations = get_label_for_pair(v1, v2, DYNAMIC_GRAPH_FILE, YEAR_END, best_ir)
                
                # 构建一行数据: [v1, v2, label, f0, f1, ..., f140]
                row = [v1, v2, label] + features
                batch_data.append(row)
                
                batch_found += 1
                total_found_in_map += 1
            else:
                batch_not_found += 1
                total_not_found += 1
            
            total_processed += 1
        
        # 保存本批数据到临时文件
        if batch_data:
            df_batch = pd.DataFrame(batch_data)
            # 设置列名
            col_names = ['v1', 'v2', 'label'] + [f'f{i}' for i in range(141)]
            df_batch.columns = col_names
            
            temp_file = os.path.join(temp_dir, f"batch_{batch_num:04d}.parquet")
            df_batch.to_parquet(temp_file, index=False)
            batch_files.append(temp_file)
            
            batch_time = time.time() - batch_start
            print(f"✅ 批次 {batch_num} 完成，耗时 {batch_time:.2f}秒，保存至 {temp_file}")
            print(f"   本批在映射中找到: {batch_found} 个，未找到: {batch_not_found} 个")
            print(f"   本批标签分布: {df_batch['label'].value_counts().to_dict()}")
            
            # 释放内存
            del df_batch, batch_data
            gc.collect()
        else:
            print(f"⚠️ 批次 {batch_num} 没有找到任何有效概念对")
    
    print(f"\n📊 统计信息:")
    print(f"   总处理概念对: {total_processed}")
    print(f"   在映射中找到: {total_found_in_map}")
    print(f"   未在映射中找到: {total_not_found}")
    print(f"   找到率: {total_found_in_map/total_processed*100:.2f}%" if total_processed > 0 else "   找到率: N/A")
    
    # 5) 合并所有批次文件
    if batch_files:
        print("\n" + "=" * 60)
        print("步骤4: 合并所有批次文件")
        print("=" * 60)
        
        # 读取并合并所有批次
        dfs = []
        for i, f in enumerate(batch_files):
            print(f"读取批次文件 {i+1}/{len(batch_files)}: {f}")
            df = pd.read_parquet(f)
            dfs.append(df)
            
            # 可选：删除临时文件
            # os.remove(f)
        
        print(f"合并 {len(dfs)} 个DataFrame...")
        df_final = pd.concat(dfs, ignore_index=True)
        
        # 打乱顺序
        df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
        
        print(f"最终数据集大小: {len(df_final)} 行, {len(df_final.columns)} 列")
        print(f"标签分布:")
        print(df_final['label'].value_counts())
        print(f"正样本比例: {df_final['label'].mean():.4f}")
        
        # 保存最终文件
        print(f"保存最终文件至: {output_file}")
        df_final.to_parquet(output_file, index=False)
        
        # 保存一个样本文件用于检查
        sample_file = output_file.replace('.parquet', '_sample.csv')
        df_final.head(1000).to_csv(sample_file, index=False)
        print(f"样本文件保存至: {sample_file}")
        
        # 清理临时文件
        print("清理临时文件...")
        for f in batch_files:
            try:
                os.remove(f)
                print(f"  已删除: {f}")
            except Exception as e:
                print(f"  删除失败 {f}: {e}")
        
        try:
            os.rmdir(temp_dir)
            print(f"  已删除临时目录: {temp_dir}")
        except Exception as e:
            print(f"  删除临时目录失败: {e}")
        
    else:
        print("❌ 没有生成任何数据批次")# generate_train_data_2022_2025.py


In [ ]:
import os
import re
import time
import torch
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pickle
import gzip
import gc
from datetime import datetime, date
import warnings
import torch.nn as nn  
import json
warnings.filterwarnings('ignore')

# ---------------- Config ----------------
BASE_DIR = r"E:\study\research\impact4cast_cscl"
RESULT_FILE = os.path.join(BASE_DIR, "2019_train", "t0_c2_result", "All_AUC_Report_Year_Train_2019.txt")
MODEL_DIR = os.path.join(BASE_DIR, "2019_train", "t0_c2_net")
OUTPUT_DIR = os.path.join(BASE_DIR, "data_for_train_2025")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 数据文件路径
FEATURE_FOLDER = os.path.join(BASE_DIR, "data_for_features")
DYNAMIC_GRAPH_FILE = os.path.join(BASE_DIR, "data_concept_graph", "full_dynamic_graph.parquet")
CONCEPT_LIST_FILE = r"E:\study\research\ASIST\entities.txt"  # 概念列表文件

# 新增：特征文件路径（包含141维归一化特征）
FEATURE_FILE = os.path.join(BASE_DIR, "data_eval", "eval_data_pair_feature.parquet")
ID_FILE = os.path.join(BASE_DIR, "data_eval", "data_eval_pair_solution.parquet")

# 时间设置
YEAR_START = 2022  # 特征基准年
YEAR_END = 2025    # 标签年
YEARS_DELTA = 3    # 时间窗口

# 分批处理参数
BATCH_SIZE = 1000000  # 每批处理的概念对数量
MAX_PAIRS = None    # 先设置为1000进行测试，完成后可以设为None处理所有

# 断点续跑相关文件
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "checkpoint.json")
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "progress.txt")

# ---------------- 模型定义（用于特征提取，如果需要） ----------------
class ff_network(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ff_network, self).__init__()
        self.semnet = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size)
        )

    def forward(self, x):
        return self.semnet(x)


# ---------------- 辅助函数 ----------------
def log_progress(message, print_to_console=True):
    """记录进度到文件"""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_message = f"[{timestamp}] {message}"
    
    if print_to_console:
        print(log_message)
    
    with open(PROGRESS_FILE, 'a', encoding='utf-8') as f:
        f.write(log_message + "\n")


def save_checkpoint(batch_num, processed_pairs, batch_files, total_found_in_map, total_not_found):
    """保存检查点信息"""
    checkpoint = {
        'batch_num': batch_num,
        'processed_pairs': processed_pairs,
        'batch_files': batch_files,
        'total_found_in_map': total_found_in_map,
        'total_not_found': total_not_found,
        'timestamp': datetime.now().isoformat()
    }
    
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(checkpoint, f, indent=2)
    
    log_progress(f"💾 检查点已保存: 批次 {batch_num}, 已处理 {processed_pairs} 个概念对")


def load_checkpoint():
    """加载检查点信息"""
    if not os.path.exists(CHECKPOINT_FILE):
        return None
    
    try:
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        log_progress(f"🔍 找到检查点: 批次 {checkpoint['batch_num']}, 已处理 {checkpoint['processed_pairs']} 个概念对")
        return checkpoint
    except Exception as e:
        log_progress(f"⚠️ 加载检查点失败: {e}")
        return None


def find_best_ir(result_file_path):
    """从结果文件中找到最优IR值（基于eval AUC）"""
    pattern = re.compile(r'IR=\[(\d+)\]: train=([\d.]+); test=([\d.]+); eval=([\d.]+)')
    results = []
    
    log_progress(f"🔎 解析结果文件: {result_file_path}")
    with open(result_file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            m = pattern.search(line)
            if m:
                ir, train, test, eval_ = m.groups()
                results.append({
                    'IR': int(ir),
                    'train': float(train),
                    'test': float(test),
                    'eval': float(eval_)
                })
    
    if not results:
        raise RuntimeError(f"未在 {result_file_path} 中解析到任何IR记录，请检查文件格式。")
    
    df = pd.DataFrame(results)
    best_idx = df['eval'].idxmax()
    best_ir = int(df.loc[best_idx, 'IR'])
    best_eval = float(df.loc[best_idx, 'eval'])
    
    log_progress(f"✅ 找到最优IR={best_ir} (eval AUC={best_eval:.4f})")
    log_progress("\n所有IR的AUC分数:")
    log_progress(df.to_string(index=False))
    
    return best_ir, best_eval, df


def load_feature_mapping(feature_file, id_file):
    """
    加载特征文件和ID文件，创建概念对到特征的映射
    
    返回:
        feature_map: 字典，键为(v1, v2)元组，值为141维特征数组
    """
    log_progress(f"🔎 加载特征映射文件...")
    
    # 加载特征文件
    if not os.path.exists(feature_file):
        raise FileNotFoundError(f"特征文件不存在: {feature_file}")
    
    log_progress(f"加载特征文件: {feature_file}")
    df_features = pd.read_parquet(feature_file)
    log_progress(f"特征文件形状: {df_features.shape}")
    
    # 加载ID文件
    if not os.path.exists(id_file):
        raise FileNotFoundError(f"ID文件不存在: {id_file}")
    
    log_progress(f"加载ID文件: {id_file}")
    df_ids = pd.read_parquet(id_file)
    
    # 确保只取需要的列
    if 'v1' in df_ids.columns and 'v2' in df_ids.columns:
        df_ids = df_ids[['v1', 'v2']]
    else:
        # 如果列名不是v1, v2，假设前两列是ID
        df_ids = df_ids.iloc[:, :2]
        df_ids.columns = ['v1', 'v2']
    
    log_progress(f"ID文件形状: {df_ids.shape}")
    
    # 检查行数是否匹配
    if len(df_features) != len(df_ids):
        raise ValueError(f"特征文件({len(df_features)}行)和ID文件({len(df_ids)}行)行数不匹配！")
    
    # 合并ID和特征
    df_combined = pd.concat([df_ids.reset_index(drop=True), 
                             df_features.reset_index(drop=True)], axis=1)
    
    # 创建映射字典
    feature_map = {}
    
    log_progress("创建概念对到特征的映射...")
    for idx, row in df_combined.iterrows():
        v1, v2 = int(row['v1']), int(row['v2'])
        # 确保v1 < v2以统一表示
        if v1 > v2:
            v1, v2 = v2, v1
        
        # 提取特征（从第3列开始）
        features = row.iloc[2:].values.astype(np.float32)
        
        # 验证特征维度
        if len(features) != 141:
            log_progress(f"⚠️ 警告: 特征维度为 {len(features)}，应为141")
        
        feature_map[(v1, v2)] = features
        
        if (idx + 1) % 1000000 == 0:
            log_progress(f"  已处理 {idx+1} 个概念对映射")
    
    log_progress(f"✅ 成功创建 {len(feature_map)} 个概念对的映射")
    
    # 显示特征统计
    if len(feature_map) > 0:
        sample_features = next(iter(feature_map.values()))
        log_progress(f"特征维度: {len(sample_features)}")
        log_progress(f"特征范围: [{sample_features.min():.4f}, {sample_features.max():.4f}]")
        log_progress(f"特征均值: {sample_features.mean():.4f}")
    
    return feature_map


def get_unconnected_pairs_batch(graph_file, year_start, batch_size=100000, max_pairs=None, start_from=0):
    """
    分批获取在year_start年未连接的概念对
    
    返回: 生成器，每批返回 (batch_pairs, total_count, current_index)
    """
    log_progress(f"🔎 正在查找 {year_start}年未连接的概念对...")
    
    # 1. 获取截至year_start的所有边
    pf = pq.ParquetFile(graph_file)
    
    # 先收集所有存在的边
    existing_pairs = set()
    total_edges = 0
    
    log_progress("正在收集已存在的概念对...")
    for batch in pf.iter_batches(batch_size=batch_size):
        df_batch = batch.to_pandas()
        
        # 检查是否有time列
        time_col = None
        for col in df_batch.columns:
            if 'time' in col.lower() or 'date' in col.lower() or 'year' in col.lower():
                time_col = col
                break
        
        # 筛选时间 <= year_start的边
        if time_col is not None:
            try:
                # 如果time_col是整数类型（如年份）
                if pd.api.types.is_integer_dtype(df_batch[time_col]):
                    df_batch = df_batch[df_batch[time_col] <= year_start]
                # 如果time_col是字符串类型
                elif pd.api.types.is_string_dtype(df_batch[time_col]):
                    year_start_str = str(year_start)
                    df_batch = df_batch[df_batch[time_col].str.startswith(year_start_str) | 
                                         (df_batch[time_col] < str(year_start + 1))]
                # 如果time_col是日期类型
                else:
                    year_start_date = pd.Timestamp(f"{year_start}-12-31")
                    df_batch = df_batch[pd.to_datetime(df_batch[time_col]) <= year_start_date]
            except Exception as e:
                log_progress(f"  时间筛选时出错: {e}，跳过时间筛选")
        
        # 添加边到集合
        for _, row in df_batch.iterrows():
            v1, v2 = row['v1'], row['v2']
            # 确保v1 < v2以统一表示
            if v1 < v2:
                existing_pairs.add((v1, v2))
            else:
                existing_pairs.add((v2, v1))
        
        total_edges += len(df_batch)
        if len(existing_pairs) % 1000000 == 0:
            log_progress(f"  已收集 {len(existing_pairs)} 个唯一概念对")
    
    log_progress(f"✅ 共收集到 {len(existing_pairs)} 个已存在的概念对（截至{year_start}年）")
    
    # 2. 获取所有概念ID
    # 从邻接矩阵或概念列表获取
    concepts = set()
    for year in [year_start, year_start-1, year_start-2]:
        cite_file = os.path.join(FEATURE_FOLDER, f"concept_node_citation_data_{year}.parquet")
        if os.path.exists(cite_file):
            try:
                df_cite = pd.read_parquet(cite_file)
                # 假设第一列是概念ID
                concepts.update(df_cite.iloc[:, 0].tolist())
            except Exception as e:
                log_progress(f"  加载 {year}年概念数据失败: {e}")
    
    concepts = list(concepts)
    if len(concepts) == 0:
        # 如果没有找到概念，从现有的边中提取
        for v1, v2 in existing_pairs:
            concepts.add(v1)
            concepts.add(v2)
        concepts = list(concepts)
    
    log_progress(f"✅ 共 {len(concepts)} 个概念")
    
    # 3. 生成所有可能的概念对
    from itertools import combinations
    
    total_possible = len(concepts) * (len(concepts) - 1) // 2
    log_progress(f"📊 所有可能的概念对总数: {total_possible:,}")
    
    # 分批生成未连接的概念对
    unconnected_batch = []
    batch_count = 0
    total_unconnected = 0
    skipped = 0
    
    for i, (v1, v2) in enumerate(combinations(concepts, 2)):
        # 如果指定了起始位置，跳过前面的组合
        if i < start_from:
            continue
        
        # 确保v1 < v2
        if v1 > v2:
            v1, v2 = v2, v1
        
        # 检查是否已存在
        if (v1, v2) not in existing_pairs:
            unconnected_batch.append((v1, v2))
            total_unconnected += 1
        
        # 每收集够batch_size或达到最大限制，就yield一批
        if len(unconnected_batch) >= batch_size or (max_pairs and total_unconnected >= max_pairs):
            if unconnected_batch:  # 确保批次不为空
                batch_count += 1
                log_progress(f"  生成批次 {batch_count}: {len(unconnected_batch)} 个未连接概念对 (从组合索引 {i+1} 开始)")
                yield unconnected_batch, total_unconnected, i + 1
                unconnected_batch = []
        
        # 如果达到最大限制，停止
        if max_pairs and total_unconnected >= max_pairs:
            break
        
        # 每处理100万个组合打印进度
        if (i + 1) % 1000000 == 0:
            log_progress(f"  已处理 {i+1:,} 个组合，找到 {total_unconnected:,} 个未连接对")
    
    # 最后一批
    if unconnected_batch:
        batch_count += 1
        log_progress(f"  生成批次 {batch_count}: {len(unconnected_batch)} 个未连接概念对")
        yield unconnected_batch, total_unconnected, i + 1
    
    log_progress(f"✅ 总共找到 {total_unconnected:,} 个未连接概念对")


def get_label_for_pair(v1, v2, graph_file, year_end, IR):
    """
    获取概念对在year_end年的标签
    
    如果该概念对在year_end年的引用数 ≥ IR，则标签=1，否则=0
    """
    try:
        pf = pq.ParquetFile(graph_file)
    except Exception as e:
        log_progress(f"  无法打开图文件: {e}", print_to_console=False)
        return 0, 0
    
    total_citations = 0
    
    # 查找包含该概念对的论文
    for batch in pf.iter_batches(batch_size=100000):
        try:
            df_batch = batch.to_pandas()
        except:
            continue
        
        # 筛选包含该概念对的论文
        mask = ((df_batch['v1'] == v1) & (df_batch['v2'] == v2)) | \
               ((df_batch['v1'] == v2) & (df_batch['v2'] == v1))
        
        if mask.any():
            df_pair = df_batch[mask]
            
            # 查找引用数列
            cite_col = None
            for col in df_pair.columns:
                if col.lower().startswith('c') and col[1:].isdigit():
                    year = int(col[1:])
                    if year <= year_end:
                        cite_col = col
                        break
                elif col.lower() in ['ct', 'citation', 'citations']:
                    cite_col = col
            
            if cite_col is not None:
                total_citations += df_pair[cite_col].sum()
    
    label = 1 if total_citations >= IR else 0
    return label, total_citations


def extract_features_from_mapping(v1, v2, feature_map):
    """
    从特征映射中提取概念对(v1, v2)的141维特征
    
    参数:
        v1, v2: 概念ID
        feature_map: 特征映射字典，键为(v1, v2)元组，值为141维特征数组
    
    返回:
        141维特征列表，如果找不到则返回None
    """
    # 确保v1 < v2以统一表示
    if v1 > v2:
        v1, v2 = v2, v1
    
    # 在映射中查找
    if (v1, v2) in feature_map:
        features = feature_map[(v1, v2)]
        return features.tolist() if isinstance(features, np.ndarray) else features
    else:
        return None


# ---------------- Main ----------------
def main():
    start_time_total = time.time()
    
    # 检查是否有检查点
    checkpoint = load_checkpoint()
    
    # 1) 找最优IR
    log_progress("=" * 60)
    log_progress("步骤1: 寻找最优IR值")
    log_progress("=" * 60)
    best_ir, best_eval, df_results = find_best_ir(RESULT_FILE)
    
    # 2) 加载特征映射
    log_progress("\n" + "=" * 60)
    log_progress("步骤2: 加载特征映射")
    log_progress("=" * 60)
    feature_map = load_feature_mapping(FEATURE_FILE, ID_FILE)
    
    # 3) 准备输出文件
    output_file = os.path.join(OUTPUT_DIR, f"train_data_{YEAR_START}_{YEAR_END}_IR{best_ir}.parquet")
    temp_dir = os.path.join(OUTPUT_DIR, "temp")
    os.makedirs(temp_dir, exist_ok=True)
    
    log_progress("\n" + "=" * 60)
    log_progress(f"步骤3: 生成训练数据文件")
    log_progress(f"输出文件: {output_file}")
    log_progress(f"特征基准年: {YEAR_START}")
    log_progress(f"标签年: {YEAR_END}")
    log_progress(f"IR阈值: {best_ir}")
    log_progress("=" * 60)
    
    # 4) 分批处理未连接的概念对
    log_progress("\n📊 开始分批处理未连接概念对...")
    
    # 从检查点恢复状态
    if checkpoint:
        start_batch = checkpoint['batch_num']
        batch_files = checkpoint['batch_files']
        total_processed = checkpoint['processed_pairs']
        total_found_in_map = checkpoint['total_found_in_map']
        total_not_found = checkpoint['total_not_found']
        log_progress(f"🔄 从检查点恢复: 从批次 {start_batch} 继续处理")
        log_progress(f"   已有临时文件: {len(batch_files)} 个")
        log_progress(f"   已处理概念对: {total_processed}")
        log_progress(f"   已找到特征: {total_found_in_map}")
        log_progress(f"   未找到特征: {total_not_found}")
    else:
        start_batch = 0
        batch_files = []
        total_processed = 0
        total_found_in_map = 0
        total_not_found = 0
    
    # 获取未连接概念对的生成器
    pair_generator = get_unconnected_pairs_batch(
        DYNAMIC_GRAPH_FILE, 
        YEAR_START, 
        batch_size=BATCH_SIZE,
        max_pairs=MAX_PAIRS
    )
    
    current_batch = 0
    for batch_pairs, total_found, current_index in pair_generator:
        current_batch += 1
        
        # 如果小于起始批次，跳过
        if current_batch <= start_batch:
            log_progress(f"⏭️ 跳过已处理的批次 {current_batch}")
            continue
        
        batch_start = time.time()
        
        log_progress(f"\n--- 处理批次 {current_batch} ---")
        log_progress(f"本批概念对数量: {len(batch_pairs)}")
        log_progress(f"累计找到概念对: {total_found}")
        log_progress(f"当前组合索引: {current_index}")
        
        # 处理本批概念对
        batch_data = []
        batch_found = 0
        batch_not_found = 0
        
        for i, (v1, v2) in enumerate(batch_pairs):
            if (i + 1) % 1000 == 0:  # 每1000个打印一次进度
                log_progress(f"  处理本批第 {i+1}/{len(batch_pairs)} 个概念对")
            
            # 从特征映射中提取特征
            features = extract_features_from_mapping(v1, v2, feature_map)
            
            if features is not None:
                # 获取标签
                label, citations = get_label_for_pair(v1, v2, DYNAMIC_GRAPH_FILE, YEAR_END, best_ir)
                
                # 构建一行数据: [v1, v2, label, f0, f1, ..., f140]
                row = [v1, v2, label] + features
                batch_data.append(row)
                
                batch_found += 1
                total_found_in_map += 1
            else:
                batch_not_found += 1
                total_not_found += 1
            
            total_processed += 1
        
        # 保存本批数据到临时文件
        if batch_data:
            df_batch = pd.DataFrame(batch_data)
            # 设置列名
            col_names = ['v1', 'v2', 'label'] + [f'f{i}' for i in range(141)]
            df_batch.columns = col_names
            
            temp_file = os.path.join(temp_dir, f"batch_{current_batch:04d}.parquet")
            df_batch.to_parquet(temp_file, index=False)
            batch_files.append(temp_file)
            
            batch_time = time.time() - batch_start
            log_progress(f"✅ 批次 {current_batch} 完成，耗时 {batch_time:.2f}秒，保存至 {temp_file}")
            log_progress(f"   本批在映射中找到: {batch_found} 个，未找到: {batch_not_found} 个")
            log_progress(f"   本批标签分布: {df_batch['label'].value_counts().to_dict()}")
            
            # 保存检查点
            save_checkpoint(current_batch, total_processed, batch_files, total_found_in_map, total_not_found)
            
            # 释放内存
            del df_batch, batch_data
            gc.collect()
        else:
            log_progress(f"⚠️ 批次 {current_batch} 没有找到任何有效概念对")
    
    log_progress(f"\n📊 统计信息:")
    log_progress(f"   总处理概念对: {total_processed}")
    log_progress(f"   在映射中找到: {total_found_in_map}")
    log_progress(f"   未在映射中找到: {total_not_found}")
    if total_processed > 0:
        log_progress(f"   找到率: {total_found_in_map/total_processed*100:.2f}%")
    
    # 5) 合并所有批次文件
    if batch_files:
        log_progress("\n" + "=" * 60)
        log_progress("步骤4: 合并所有批次文件")
        log_progress("=" * 60)
        
        # 读取并合并所有批次
        dfs = []
        for i, f in enumerate(batch_files):
            log_progress(f"读取批次文件 {i+1}/{len(batch_files)}: {f}")
            df = pd.read_parquet(f)
            dfs.append(df)
        
        log_progress(f"合并 {len(dfs)} 个DataFrame...")
        df_final = pd.concat(dfs, ignore_index=True)
        
        # 打乱顺序
        df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
        
        log_progress(f"最终数据集大小: {len(df_final)} 行, {len(df_final.columns)} 列")
        log_progress(f"标签分布:")
        log_progress(str(df_final['label'].value_counts()))
        log_progress(f"正样本比例: {df_final['label'].mean():.4f}")
        
        # 保存最终文件
        log_progress(f"保存最终文件至: {output_file}")
        df_final.to_parquet(output_file, index=False)
        
        # 保存一个样本文件用于检查
        sample_file = output_file.replace('.parquet', '_sample.csv')
        df_final.head(1000).to_csv(sample_file, index=False)
        log_progress(f"样本文件保存至: {sample_file}")
        
        # 清理临时文件
        log_progress("清理临时文件...")
        for f in batch_files:
            try:
                os.remove(f)
                log_progress(f"  已删除: {f}")
            except Exception as e:
                log_progress(f"  删除失败 {f}: {e}")
        
        try:
            os.rmdir(temp_dir)
            log_progress(f"  已删除临时目录: {temp_dir}")
        except Exception as e:
            log_progress(f"  删除临时目录失败: {e}")
        
        # 删除检查点文件（处理完成）
        if os.path.exists(CHECKPOINT_FILE):
            os.remove(CHECKPOINT_FILE)
            log_progress("✅ 已删除检查点文件")
        
    else:
        log_progress("❌ 没有生成任何数据批次")
    
    total_time = time.time() - start_time_total
    log_progress("\n" + "=" * 60)
    log_progress(f"✅ 全部完成！总耗时: {total_time:.2f}秒")
    log_progress(f"输出文件: {output_file}")
    log_progress("=" * 60)


if __name__ == "__main__":
    main()
    
    total_time = time.time() - start_time_total
    print("\n" + "=" * 60)
    print(f"✅ 全部完成！总耗时: {total_time:.2f}秒")
    print(f"输出文件: {output_file}")
    print("=" * 60)


if __name__ == "__main__":
    main()

In [ ]:
import os
import re
import time
import torch
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pickle
import gzip
import gc
from datetime import datetime, date
import warnings
import torch.nn as nn  
import json
warnings.filterwarnings('ignore')

# ---------------- Config ----------------
BASE_DIR = r"E:\study\research\impact4cast_cscl"
RESULT_FILE = os.path.join(BASE_DIR, "2016_train", "t0_c2_result", "All_AUC_Report_Year_Train_2016.txt")
MODEL_DIR = os.path.join(BASE_DIR, "2016_train", "t0_c2_net")
OUTPUT_DIR = os.path.join(BASE_DIR, "data_for_train")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 数据文件路径
FEATURE_FOLDER = os.path.join(BASE_DIR, "data_for_features")
DYNAMIC_GRAPH_FILE = os.path.join(BASE_DIR, "data_concept_graph", "full_dynamic_graph.parquet")
CONCEPT_LIST_FILE = r"E:\study\research\ASIST\entities.txt" # 概念列表文件

# 新增：特征文件路径（包含141维归一化特征）
FEATURE_FILE = os.path.join(BASE_DIR, "data_eval", "eval_data_pair_feature.parquet")
ID_FILE = os.path.join(BASE_DIR, "data_eval", "data_eval_pair_solution.parquet")

# 时间设置
YEAR_START = 2016  # 特征基准年
YEAR_END = 2019    # 标签年
YEARS_DELTA = 3    # 时间窗口

# 分批处理参数
BATCH_SIZE = 1000000  # 每批处理的概念对数量
MAX_PAIRS = None    # 先设置为1000进行测试，完成后可以设为None处理所有

# 断点续跑相关文件
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "checkpoint.json")
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "progress.txt")

# ---------------- 模型定义（用于特征提取，如果需要） ----------------
class ff_network(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ff_network, self).__init__()
        self.semnet = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size)
        )

    def forward(self, x):
        return self.semnet(x)


# ---------------- 辅助函数 ----------------
def log_progress(message, print_to_console=True):
    """记录进度到文件"""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_message = f"[{timestamp}] {message}"
    
    if print_to_console:
        print(log_message)
    
    with open(PROGRESS_FILE, 'a', encoding='utf-8') as f:
        f.write(log_message + "\n")


def save_checkpoint(batch_num, processed_pairs, batch_files, total_found_in_map, total_not_found):
    """保存检查点信息"""
    checkpoint = {
        'batch_num': batch_num,
        'processed_pairs': processed_pairs,
        'batch_files': batch_files,
        'total_found_in_map': total_found_in_map,
        'total_not_found': total_not_found,
        'timestamp': datetime.now().isoformat()
    }
    
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(checkpoint, f, indent=2)
    
    log_progress(f"💾 检查点已保存: 批次 {batch_num}, 已处理 {processed_pairs} 个概念对")


def load_checkpoint():
    """加载检查点信息"""
    if not os.path.exists(CHECKPOINT_FILE):
        return None
    
    try:
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        log_progress(f"🔍 找到检查点: 批次 {checkpoint['batch_num']}, 已处理 {checkpoint['processed_pairs']} 个概念对")
        return checkpoint
    except Exception as e:
        log_progress(f"⚠️ 加载检查点失败: {e}")
        return None


def find_best_ir(result_file_path):
    """从结果文件中找到最优IR值（基于eval AUC）"""
    pattern = re.compile(r'IR=\[(\d+)\]: train=([\d.]+); test=([\d.]+); eval=([\d.]+)')
    results = []
    
    log_progress(f"🔎 解析结果文件: {result_file_path}")
    with open(result_file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            m = pattern.search(line)
            if m:
                ir, train, test, eval_ = m.groups()
                results.append({
                    'IR': int(ir),
                    'train': float(train),
                    'test': float(test),
                    'eval': float(eval_)
                })
    
    if not results:
        raise RuntimeError(f"未在 {result_file_path} 中解析到任何IR记录，请检查文件格式。")
    
    df = pd.DataFrame(results)
    best_idx = df['eval'].idxmax()
    best_ir = int(df.loc[best_idx, 'IR'])
    best_eval = float(df.loc[best_idx, 'eval'])
    
    log_progress(f"✅ 找到最优IR={best_ir} (eval AUC={best_eval:.4f})")
    log_progress("\n所有IR的AUC分数:")
    log_progress(df.to_string(index=False))
    
    return best_ir, best_eval, df


def load_feature_mapping(feature_file, id_file):
    """
    加载特征文件和ID文件，创建概念对到特征的映射
    
    返回:
        feature_map: 字典，键为(v1, v2)元组，值为141维特征数组
    """
    log_progress(f"🔎 加载特征映射文件...")

    # 加载特征文件
    if not os.path.exists(feature_file):
        raise FileNotFoundError(f"特征文件不存在: {feature_file}")
    
    log_progress(f"加载特征文件: {feature_file}")
    df_features = pd.read_parquet(feature_file)
    log_progress(f"特征文件形状: {df_features.shape}")
    
    # 加载ID文件
    if not os.path.exists(id_file):
        raise FileNotFoundError(f"ID文件不存在: {id_file}")
    
    log_progress(f"加载ID文件: {id_file}")
    df_ids = pd.read_parquet(id_file)
    
    # 确保只取需要的列
    if 'v1' in df_ids.columns and 'v2' in df_ids.columns:
        df_ids = df_ids[['v1', 'v2']]
    else:
        # 如果列名不是v1, v2，假设前两列是ID
        df_ids = df_ids.iloc[:, :2]
        df_ids.columns = ['v1', 'v2']
    
    log_progress(f"ID文件形状: {df_ids.shape}")
    
    # 检查行数是否匹配
    if len(df_features) != len(df_ids):
        raise ValueError(f"特征文件({len(df_features)}行)和ID文件({len(df_ids)}行)行数不匹配！")
    
    # 合并ID和特征
    df_combined = pd.concat([df_ids.reset_index(drop=True), 
                             df_features.reset_index(drop=True)], axis=1)
    
    # 创建映射字典
    feature_map = {}
    
    log_progress("创建概念对到特征的映射...")
    for idx, row in df_combined.iterrows():
        v1, v2 = int(row['v1']), int(row['v2'])
        # 确保v1 < v2以统一表示
        if v1 > v2:
            v1, v2 = v2, v1
        
        # 提取特征（从第3列开始）
        features = row.iloc[2:].values.astype(np.float32)
        
        # 验证特征维度
        if len(features) != 141:
            log_progress(f"⚠️ 警告: 特征维度为 {len(features)}，应为141")
        
        feature_map[(v1, v2)] = features
        
        if (idx + 1) % 1000000 == 0:
            log_progress(f"  已处理 {idx+1} 个概念对映射")
    
    log_progress(f"✅ 成功创建 {len(feature_map)} 个概念对的映射")
    
    # 显示特征统计
    if len(feature_map) > 0:
        sample_features = next(iter(feature_map.values()))
        log_progress(f"特征维度: {len(sample_features)}")
        log_progress(f"特征范围: [{sample_features.min():.4f}, {sample_features.max():.4f}]")
        log_progress(f"特征均值: {sample_features.mean():.4f}")
    
    return feature_map


def get_labels_from_solution_files(unconnected_connected_file, unconnected_unconnected_file, feature_map):
    """
    从未连接的概念对文件中生成标签数据，连接2019年的概念对标签为1，未连接的标签为0。
    """
    # 载入2016年未连接但在2019年连接的概念对
    connected_df = pd.read_parquet(unconnected_connected_file)
    connected_pairs = set(zip(connected_df['v1'], connected_df['v2']))

    # 载入2016年和2019年均未连接的概念对
    unconnected_df = pd.read_parquet(unconnected_unconnected_file)
    unconnected_pairs = set(zip(unconnected_df['v1'], unconnected_df['v2']))

    # 确保在映射中都存在对应特征的概念对
    labels = []
    for v1, v2 in connected_pairs:
        features = extract_features_from_mapping(v1, v2, feature_map)
        if features is not None:
            labels.append([v1, v2, 1] + features.tolist())

    for v1, v2 in unconnected_pairs:
        features = extract_features_from_mapping(v1, v2, feature_map)
        if features is not None:
            labels.append([v1, v2, 0] + features.tolist())

    return labels

# 修改主函数，按照要求输出数据集
def main():
    start_time_total = time.time()
    
    # 检查是否有检查点
    checkpoint = load_checkpoint()
    
    # 1) 找最优IR
    log_progress("=" * 60)
    log_progress("步骤1: 寻找最优IR值")
    log_progress("=" * 60)
    best_ir, best_eval, df_results = find_best_ir(RESULT_FILE)
    
    # 2) 加载特征映射
    log_progress("\n" + "=" * 60)
    log_progress("步骤2: 加载特征映射")
    log_progress("=" * 60)
    feature_map = load_feature_mapping(FEATURE_FILE, ID_FILE)
    
    # 3) 从文件中生成标签数据
    log_progress("\n" + "=" * 60)
    log_progress("步骤3: 生成标签数据")
    log_progress("=" * 60)
    unconnected_connected_file = os.path.join(BASE_DIR, "data_pair_solution", "unconnected_2016_pair_solution_connected_2019.parquet")
    unconnected_unconnected_file = os.path.join(BASE_DIR, "data_pair_solution", "unconnected_2016_pair_solution_unconnected_2019.parquet")
    
    labels_data = get_labels_from_solution_files(unconnected_connected_file, unconnected_unconnected_file, feature_map)
    
    # 4) 保存数据到文件
    output_file = os.path.join(OUTPUT_DIR, f"train_data_{YEAR_START}_{YEAR_END}_IR{best_ir}.parquet")
    log_progress(f"输出文件: {output_file}")
    
    df_labels = pd.DataFrame(labels_data)
    col_names = ['v1', 'v2', 'label'] + [f'f{i}' for i in range(141)]
    df_labels.columns = col_names
    
    df_labels.to_parquet(output_file, index=False)
    
    log_progress(f"✅ 生成数据文件成功，保存至 {output_file}")
    
    total_time = time.time() - start_time_total
    log_progress("\n" + "=" * 60)
    log_progress(f"✅ 全部完成！总耗时: {total_time:.2f}秒")
    log_progress("=" * 60)


if __name__ == "__main__":
    main()
    
    total_time = time.time() - start_time_total
    print("\n" + "=" * 60)
    print(f"✅ 全部完成！总耗时: {total_time:.2f}秒")
    print("=" * 60)